### Imports
Please follow the README's instructions under 'Installations' to ensure you have the appropriate dependencies to run this notebook. 

In [ ]:
# Standard library
import collections
import glob
import os
import re
import shutil
import time

# Third-party
import cv2
import keras
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import PIL
import skimage.io as io
import tensorflow as tf
from PIL import Image, ImageDraw, ImageEnhance, ImageFont
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

# Keras / TensorFlow Keras
from keras import backend, optimizers, utils
from keras.applications import VGG19
from keras.callbacks import ModelCheckpoint, TensorBoard
from keras.layers import Activation, Dense, Dropout, Flatten
from keras.models import Sequential
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import keras
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import PIL
import skimage.io as io
import tensorflow as tf
from PIL import Image, ImageDraw, ImageEnhance, ImageFont
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

# Keras / TensorFlow Keras
from keras import backend, optimizers, utils
from keras.applications import VGG19
from keras.callbacks import ModelCheckpoint, TensorBoard
from keras.layers import Activation, Dense, Dropout, Flatten
from keras.models import Sequential
from keras.preprocessing import image
from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

### Utils

In [ ]:
def constructVGGModel(modelPath):
    dx=dy=48
    res_conv = VGG19(weights='imagenet', include_top=False, input_shape=(dx, dy, 3))
    
    # Freeze the layers except the last 12 layers
    for layer in res_conv.layers[:-12]:
        layer.trainable = False
    
    # Create the model
    global model1
    model1 = Sequential()
   
    # Add the res convolutional base model
    model1.add(res_conv)
    
    # Add new layers
    model1.add(Flatten())
    model1.add(Dense(1024, activation='relu'))
    model1.add(Dropout(0.2))
    model1.add(Dense(3, activation='softmax'))
    model1.load_weights(modelPath)
    return model1

# Function to split data
def split_data(class_name, source_dir, train_dir, val_dir, test_dir, val_split=0.2, test_split=0.1):
    class_dir = os.path.join(source_dir, class_name)
    images = os.listdir(class_dir)
    train_images, val_images = train_test_split(images, test_size=val_split)
    train_images, test_images = train_test_split(train_images, test_size=test_split / (1 - val_split))

    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for image in train_images:
        shutil.copy(os.path.join(class_dir, image), os.path.join(train_dir, class_name, image))
    
    for image in val_images:
        shutil.copy(os.path.join(class_dir, image), os.path.join(val_dir, class_name, image))

    for image in test_images:
        shutil.copy(os.path.join(class_dir, image), os.path.join(test_dir, class_name, image))

### Constructing + Loading the Model

In [ ]:
# define the path to the model you want to re-train
modelPath = r"path/to/model.h5" # TODO

# loading + compiling the model
print("Original model summary:")
try:
    model = constructVGGModel(modelPath)
except Exception as e:
    print(f"constructVGGModel failed: {e}\nFalling back to load_model...")
    model = load_model(modelPath)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # or your preferred optimizer
    loss='categorical_crossentropy',                         # or 'sparse_categorical_crossentropy' if using integer labels
    metrics=['accuracy']
)

# freeze all layers except the last few layers for transfer learning to new training annotations
for layer in model.layers[:-4]: # Adjust the slicing based on which layers you want to fine-tune
    layer.trainable = False

# print the model summary after freezing layers
print("Model summary after freezing layers:")
model.summary()


### Train, Test, and Validation Data Preparation

Before running the data preparation steps, all sampled training patches must be organized into a single source directory (data_dir) with one subfolder per annotation category. The subfolder names define the class labels and must be consistent across runs — for example, stroma, epithelium, and others. All .tif patch images for a given category should be placed flat inside their respective subfolder (no nested subdirectories). The split_data function will then automatically partition each class into train/, val/, and test/ splits using the specified ratios, mirroring the same subfolder structure within each split directory. The expected layout going in is:


In [ ]:
# data_dir/
# ├── stroma/
# │   ├── patch_001.tif
# │   └── ...
# ├── epithelium/
# │   ├── patch_001.tif
# │   └── ...
# └── others/
#     ├── patch_001.tif
#     └── ...

In [ ]:
# TODO: Define file paths
data_dir = r"path/to/training/annotations"
train_dir = r"path/to/train/split"
val_dir = r"path/to/val/split"
test_dir = r"path/to/test/split"

In [ ]:
# Create directories for train, validation, and test sets
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Split the data for each class
for class_name in os.listdir(data_dir):
    split_data(class_name, data_dir, train_dir, val_dir, test_dir)


Once all the training annotations have been split, we will create dataloaders to implement augmentation for subsequent training. Below are the defaults used in the training of our models.

In [ ]:
# Create an ImageDataGenerator for the training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,             # Rotating the image (old:20)
    shear_range=0.2,               # Shearing
    zoom_range=0.2,                # Zooming in/out
    horizontal_flip=True,          # Horizontal flipping
    vertical_flip=True,            # Vertical flipping
    brightness_range=[0.8, 1.2],   # Brightness jitter (new)
    width_shift_range=0.2,         # Shifting horizontally
    height_shift_range=0.2         # Shifting vertically
)

# Create an ImageDataGenerator for the validation and test data (only rescaling)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create the train generator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(48, 48),    # Resize images to 48x48 pixels
    batch_size=32,
    class_mode='categorical'
)

# Create the validation generator
val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(48, 48),    # Resize images to 48x48 pixels
    batch_size=32,
    class_mode='categorical'
)

# Create the test generator
test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(48, 48),    # Resize images to 48x48 pixels
    batch_size=32,
    class_mode='categorical',
    shuffle=False            # Important for evaluation and predictions
)

# Print the class indices to ensure they match your expected class_to_idx mapping
print(train_generator.class_indices)
print(val_generator.class_indices)
print(test_generator.class_indices)


### Training the Model

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, verbose=1, restore_best_weights=True),
    ModelCheckpoint('path/to/trained/model.h5', monitor='val_accuracy', save_best_only=True, verbose=1), # TODO: define the name of your new model
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.1, patience=5, verbose=1)
]

# Train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,  # Start with a baseline number of epochs
    steps_per_epoch=len(train_generator),
    validation_steps=len(val_generator),
    callbacks=callbacks
)

# Plot training and validation curves
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.show()
